# Auto-R diagnostic — how many references does $f$ actually need?

The FLUX sweeps so far used **R = 4** hand-picked references. That is the knob I most suspect of explaining
why the tilt moved $f$ by ≈2σ yet produced only ordinary dogs: with R=4,
$f(x) = E_{x' \sim p}[D^{2}_{IEM}(x,x')]$ is really *"how far is $x$ from these four particular dogs"*, and
plenty of perfectly ordinary dogs are far from four specific ones. If that is what is happening, raising λ
cannot help — the reward is not measuring novelty relative to $p$.

**The repo already has the principled test.** `RefSelector.select(R=None)` sweeps a doubling ladder and accepts
the smallest R whose **weighted-τ** rank correlation $\tau(f_R, f_{2R}) \geq$ `tau_target` — i.e. where
doubling the references stops changing how $f$ *ranks* samples. τ is unitless, so the bar transfers across
dimensions, though CLAUDE.md warns that in high $d$ the achievable **ceiling** may sit below 0.97;
`tau_history` exposes that ceiling directly.

**Both selectors, because they fail differently.**

- `RandomRefs` — i.i.d. draws, uniform weights. *Unbiased* for $E_{x'\sim p}[D]$, variance $\propto 1/R$. Its
  auto-R averages over independent reference draws, since draw-to-draw variance is exactly what sets R.
- `WeightedFPSRefs` — greedy farthest-point coverage in the IEM metric, each reference weighted by the
  $p$-mass of its Voronoi cell. *Consistent* rather than unbiased, and it covers the space rather than
  sampling it. Its own docstring warns the auto-chosen R is **optimistic**: the nested R-vs-2R test shares R
  reference terms, an intrinsic bias of deterministic FPS. **Treat its R as a lower bound.**

Agreement between the two is therefore real evidence; disagreement localises the problem.

## Chosen settings, and why

At the measured 174 ms per single-image transformer pass, the library defaults are unrunnable at FLUX scale:

| configuration | cost |
|---|---|
| `RandomRefs` defaults (probes=128, draws=20, grid≤32) | ≈120 h |
| `WeightedFPSRefs` defaults (pool=1000, est=1000, grid≤128) | ≈558 h |
| **the settings below** | **≈4.7 h** |

The blow-up is structural: `RandomRefs` pays `draws × pairwise(probes, 2·max_R)`, while the FPS family
additionally samples a **pool** and runs one IEM `pairwise(pool, 1)` per greedy pick, then two more
independent estimation sets for the Voronoi weights.

| knob | value | reason |
|---|---|---|
| `N_GAMMA`, `NUM_EPS` | 6, 3 | pinned to the sweep, so the answer applies to *that* reward |
| `GRID` | (1,2,4,8) | gives τ at R=4 (f₄ vs f₈) *and* R=8; extending to 16 would cost ≈10.8 h |
| `PROBE_SIZE` | **64** | τ resolution — see below. This is the one knob not minimised |
| `RAND_DRAWS` | 3 | averages real reference-draw variance; 20 is unaffordable |
| `FPS_POOL` | 64 | ≥ 2·max(GRID); picking 16 of 64 keeps "farthest-point" meaningful |
| `TAU_TARGET` | 0.97 | nominal bar, with `fallback="best"` so a lower ceiling reports instead of raising |

**Why `PROBE_SIZE` is not minimised.** Kendall-τ sampling SE under independence is
$\sqrt{2(2n+5)/(9n(n-1))}$: **0.185** at n=16, 0.124 at 32, **0.086** at 64, 0.060 at 128. At 16 probes a
single τ cannot separate 0.80 from 0.97 — the diagnostic would return a number too noisy to act on. 64 is the
cheapest defensible choice. (The real SE is smaller, since $f_R$ and $f_{2R}$ are nested and strongly
correlated, but that is what rules out 16.)

**Consequently: read the τ *curve*, not a single value against 0.97.** Whether τ rises with R, and where it
flattens, is robust; "is τ=0.95 below target?" is not.

## How to read the result

- **High τ already at R=4** → references are not the bottleneck. Spend the budget on `GUIDANCE` (free, and it
  sets how wide $p$ is — the manifold the tilt is confined to) and on `N_GAMMA`.
- **Low τ at R=4, rising with R** → R=4 was distorting `f`, and the expensive increase is justified by
  measurement rather than suspicion. Budget consequence at 24 sweeps/level: R=8 buys ~16.7 levels in 24 h,
  R=16 only ~8.8, and R=32 does not reach β=1 in any allocation available here.

In [ ]:
import json
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

T_START = time.time()
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(REPO, "creativity_measure")):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)
OUT_DIR = os.path.join(REPO, "notebooks", "refset_auto_r")
os.makedirs(OUT_DIR, exist_ok=True)

from creativity_measure import set_default_dtype

set_default_dtype(torch.float32)
torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16

from huggingface_hub import auth_check

auth_check("black-forest-labs/FLUX.1-dev")

# --- model + prompt: identical to the sweeps, so the answer applies to THAT reward ---------------
from diffusers import FluxPipeline

MODEL_ID, PROMPT, GUIDANCE = "black-forest-labs/FLUX.1-dev", "A dog", 3.5
IMG = 512
C, H, W = 16, IMG // 8, IMG // 8
LATENT_DIM = C * H * W
SIGMA_MIN, SIGMA_MAX, N_STEPS = 2e-3, 80.0, 8

pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
with torch.no_grad():
    prompt_embeds, pooled_prompt_embeds, text_ids = pipe.encode_prompt(
        prompt=PROMPT, prompt_2=None, device=DEVICE, max_sequence_length=512)
pipe.text_encoder = pipe.text_encoder_2 = pipe.tokenizer = pipe.tokenizer_2 = None
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
transformer = pipe.transformer.eval()

from creativity_measure.distances.edm_adapter import edm_score_fn
from creativity_measure.generators.base import edm_generator

img_ids = FluxPipeline._prepare_latent_image_ids(1, H // 2, W // 2, DEVICE, DTYPE)
txt_ids = text_ids if text_ids.ndim == 2 else text_ids[0]


def denoiser(x: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """FLUX velocity -> EDM denoiser E[X|x_sigma].  t = sigma/(1+sigma), x_t = x_sigma/(1+sigma),
    x_0 = x_t - t*v (exact). fp32 out; validated by the baseline run's lambda_eff=0 row."""
    b = x.shape[0]
    sig = sigma.reshape(b, 1, 1, 1).to(torch.float32)
    t = (sigma / (1.0 + sigma)).to(torch.float32)
    x_t = x.to(torch.float32) / (1.0 + sig)
    with torch.no_grad():
        v = transformer(
            hidden_states=FluxPipeline._pack_latents(x_t.to(DTYPE), b, C, H, W),
            timestep=t.to(DTYPE),
            guidance=torch.full((b,), GUIDANCE, device=x.device, dtype=torch.float32),
            pooled_projections=pooled_prompt_embeds.expand(b, -1).to(DTYPE),
            encoder_hidden_states=prompt_embeds.expand(b, -1, -1).to(DTYPE),
            txt_ids=txt_ids, img_ids=img_ids, return_dict=False)[0]
    v = FluxPipeline._unpack_latents(v, IMG, IMG, 8).to(torch.float32)
    return (x_t - t.reshape(b, 1, 1, 1) * v).to(torch.float32)


G = edm_generator(denoiser, img_shape=(C, H, W), sigma_min=SIGMA_MIN,
                  sigma_max=SIGMA_MAX, n_steps=N_STEPS)
score_fn = edm_score_fn(denoiser, (C, H, W))
print(f"ready on {DEVICE}   d={LATENT_DIM}   prompt={PROMPT!r} guidance={GUIDANCE}")

In [ ]:
from creativity_measure import SquaredGlobalIEMDistance

# --- IEM knobs: EXACTLY the sweep's, so the answer applies to that reward ------------------------
N_GAMMA, NUM_EPS = 6, 3

# =================================================================================================
# AUTO-R CONTROLS
#
# COST IS SET BY draws * max(GRID). _auto_select_R builds one tau block of 2*max(GRID) references PER
# DRAW, up front, before the R loop; the loop then only column-slices them. Consequences:
#   * adding small R values to a grid is FREE -- same block;
#   * lowering tau_target saves NOTHING -- the early exit skips only cheap tau computations;
#   * draws trade DIRECTLY against range: draws=3/maxR=16 costs the same as draws=1/maxR=64.
#
# WHAT THIS DIAGNOSTIC IS ACTUALLY FOR. Finding the R that i.i.d. references need may be both
# unanswerable here and irrelevant: R=16 already buys only ~8.8 SMC levels in 24 h, and R=32 does not
# reach beta=1 in any allocation available. The decidable question is which SELECTOR reaches adequate
# rank stability at an AFFORDABLE R -- and there is an economic asymmetry that makes that the right
# question:
#
#     WeightedFPSRefs is expensive to CONSTRUCT (pool + greedy + Voronoi weights) but FREE TO USE.
#     Once refs and weights are frozen, a reward call is a weighted mean over R references -- exactly
#     the cost of RandomRefs at the same R. Construction is paid ONCE, outside the SMC loop.
#
# So if WeightedFPS at R=8 ranks as well as RandomRefs at R=64, the sweep gets R=64-quality reward at
# R=8 sampling cost. "tau still rising at the top of the RandomRefs ladder" is therefore a FINDING,
# not a failure: it says i.i.d. references are hopeless at d=65536 and points at the coverage-based
# selector. Hence the asymmetric ladders below.
# =================================================================================================
RAND_GRID = (4, 8, 16, 32)   # tau at R=4,8,16,32 -> 64 refs. Establishes the trend and whether the
                             # i.i.d. estimator plateaus anywhere reachable.   (~6.2 h at draws=2)
RAND_DRAWS = 2               # 2 draws to R=32 beats 1 draw to R=64 at equal cost: RandomRefs' premise
                             # is that draw-to-draw variance sets R, and one draw cannot show that.
FPS_GRID = (4, 8)            # the cross-check arm: does coverage+weights get there far sooner?  (~2.4 h)

# PROBE_SIZE is deliberately NOT minimal: it sets how well tau can be resolved at all.
# Kendall-tau sampling SE under independence is sqrt(2(2n+5)/(9n(n-1))):
#     n=16 -> 0.185     n=32 -> 0.124     n=64 -> 0.086     n=128 -> 0.060
# At 16 probes a single tau cannot separate 0.80 from 0.97, which would make this unactionable.
# The true SE is smaller -- f_R and f_2R are nested and strongly correlated -- but that rules out 16.
PROBE_SIZE = 64

# FPS_POOL must be >> 2*max(FPS_GRID), not merely >=: at 2*max == pool, FPS selects the ENTIRE pool,
# so f_2R stops being an independent reference set and that tau becomes meaningless. 16 of 64 is fine.
FPS_POOL = 64
FPS_PPC, FPS_FLOOR = 4, 32   # Voronoi-weight estimation points (defaults 64 / 1000)

# 0.9 rather than the library's 0.97: CLAUDE.md warns the achievable tau ceiling falls in high d, and
# at 64 probes a 0.9 bar is resolvable while 0.97 is not. fallback="best" reports rather than raises.
TAU_TARGET = 0.9
SEED = 17


class GeneratorSampler:
    """Adapts G(z) to the `.sample(n, seed=...)` interface RefSelector expects.

    The selectors only need a sampler -- never log p -- so no `Density` is required. Latents come from
    a CPU generator then move to the device, matching how the sweep notebooks draw z.
    """

    def __init__(self, G, d, device):
        self.G, self.d, self.device = G, d, device

    def sample(self, n: int, seed: int | None = None):
        gen = torch.Generator(device="cpu").manual_seed(0 if seed is None else int(seed))
        with torch.no_grad():
            return self.G(torch.randn(n, self.d, generator=gen).to(self.device))


p_gen = GeneratorSampler(G, LATENT_DIM, DEVICE)

# gammas must exist before the Distance is built (the constructor is annotated for a real tensor).
# GAMMA_LO = 1/S^2 from the data length scale: beyond sigma = S the observation is pure noise, so
# there is no signal to add. Same rule as the sweep notebooks, so the metric is the same one.
_probe = p_gen.sample(4, seed=SEED)
S = _probe.std().item()
GAMMA_LO = max(1.0 / S**2, 1.0 / SIGMA_MAX**2)
GAMMA_HI = min(2.0**10, 1.0 / SIGMA_MIN**2)
GAMMAS = torch.logspace(math.log2(GAMMA_LO), math.log2(GAMMA_HI), N_GAMMA, base=2).to(DEVICE)
D2 = SquaredGlobalIEMDistance(None, GAMMAS, num_eps=NUM_EPS, seed=123, score_fn=score_fn)
print(f"data scale S={S:.3f}   gamma=[{GAMMA_LO:.3g}, {GAMMA_HI:.3g}] x {N_GAMMA} nodes")

# --- cost projection (report only; no abort) ------------------------------------------------------
S_PASS = 0.174     # measured seconds per single-image pass, from job 693836
pw = lambda K, M: M * (N_GAMMA - 1) * (NUM_EPS + NUM_EPS * K)   # pairwise(K rows, M refs), in passes
gp = lambda n: (2 * N_STEPS - 1) * n                            # G(n)

_rmax = max(RAND_GRID)
rand_passes = gp(PROBE_SIZE) + RAND_DRAWS * (gp(2 * _rmax) + pw(PROBE_SIZE, 2 * _rmax))
_fmax = max(FPS_GRID)
_est = max(FPS_FLOOR, FPS_PPC * _fmax)
fps_passes = (gp(FPS_POOL) + 2 * _fmax * pw(FPS_POOL, 1)
              + 2 * (gp(_est) + pw(_est, 2 * _fmax)) + pw(PROBE_SIZE, 2 * _fmax))
print(f"\nprojected  RandomRefs  grid<={_rmax:2d} ({2 * _rmax:3d} refs, {RAND_DRAWS} draws): "
      f"{rand_passes:>8,} passes = {rand_passes * S_PASS / 3600:5.2f} h")
print(f"           WeightedFPS grid<={_fmax:2d} ({2 * _fmax:3d} refs, pool {FPS_POOL}):    "
      f"{fps_passes:>8,} passes = {fps_passes * S_PASS / 3600:5.2f} h")
print(f"           total {(rand_passes + fps_passes) * S_PASS / 3600:.2f} h   (--time=600 allows 10 h)")

assert 2 * _fmax < FPS_POOL, (
    f"FPS needs pool_size STRICTLY greater than 2*max(FPS_GRID)={2 * _fmax}; at equality FPS selects "
    f"the whole pool and the largest-R tau is meaningless. Got pool={FPS_POOL}.")

RESULTS_PATH = os.path.join(OUT_DIR, "auto_r_results.json")
results: dict[str, dict] = {}


def save(tag: str, payload: dict) -> None:
    """Persist after each selector, so a timeout still leaves the finished one on disk."""
    results[tag] = payload
    tmp = RESULTS_PATH + ".tmp"
    with open(tmp, "w") as fh:
        json.dump({"config": {"N_GAMMA": N_GAMMA, "NUM_EPS": NUM_EPS,
                              "rand_grid": list(RAND_GRID), "fps_grid": list(FPS_GRID),
                              "probe_size": PROBE_SIZE, "rand_draws": RAND_DRAWS,
                              "fps_pool": FPS_POOL, "fps_ppc": FPS_PPC, "fps_floor": FPS_FLOOR,
                              "tau_target": TAU_TARGET, "seed": SEED, "prompt": PROMPT,
                              "guidance": GUIDANCE, "d": LATENT_DIM, "data_scale_S": S},
                   "results": results}, fh, indent=2)
    os.replace(tmp, RESULTS_PATH)
    print(f"  saved -> {os.path.basename(RESULTS_PATH)}")

In [ ]:
from creativity_measure.refset import RandomRefs

# RandomRefs first: it is the cheaper of the two AND the unbiased estimator, so it gets the wider
# ladder (RAND_GRID) and its result survives if the job is cut short.
# fallback="best" so a tau ceiling below TAU_TARGET reports the best R instead of raising -- CLAUDE.md
# warns the achievable ceiling in high d may sit under the bar, and that ceiling is itself the finding.
sel_rand = RandomRefs(
    p_gen, distance=D2, seed=SEED,
    auto_r_grid=RAND_GRID, tau_target=TAU_TARGET, probe_size=PROBE_SIZE,
    auto_r_draws=RAND_DRAWS, fallback="best",
)

t0 = time.time()
R_rand = sel_rand.select(None).shape[0]
dt_rand = time.time() - t0

print(f"RandomRefs auto-R -> R = {R_rand}   ({dt_rand / 60:.1f} min, "
      f"projected {rand_passes * S_PASS / 60:.0f} min)")
print(f"{'R':>4} {'tau(f_R, f_2R)':>15}  {'>= target':>9}")
for R, tau in sel_rand.tau_history:
    print(f"{R:4d} {tau:15.4f}  {'yes' if tau >= TAU_TARGET else 'no':>9}")
ceiling_rand = max((t for _, t in sel_rand.tau_history), default=float("nan"))
print(f"\nobserved tau ceiling: {ceiling_rand:.4f}   (target {TAU_TARGET})")
tau_at_4 = dict(sel_rand.tau_history).get(4)
if tau_at_4 is not None:
    verdict = ("STABLE: R=4 was adequate" if tau_at_4 >= TAU_TARGET else
               "UNSTABLE: doubling to 8 reorders f, so R=4 was distorting the reward")
    print(f"tau at R=4 (the sweeps' choice): {tau_at_4:.4f}  -> {verdict}")

save("random_refs", {"R": R_rand, "tau_history": sel_rand.tau_history,
                     "ceiling": ceiling_rand, "minutes": dt_rand / 60,
                     "grid": list(RAND_GRID)})

In [ ]:
from creativity_measure.refset import WeightedFPSRefs

# WeightedFPSRefs: deterministic given the pool, so its auto-R runs ONE draw -- but it pays for a
# sampled pool, one IEM pairwise(pool, 1) per greedy pick, and two independent estimation sets for the
# Voronoi weights. That is why it gets the narrower FPS_GRID; `start=0` avoids an outlier search.
#
# Its docstring is explicit that the auto-chosen R is OPTIMISTIC: the nested R-vs-2R test shares R
# reference terms, which is intrinsic to deterministic FPS and not removable. Read its R as a LOWER
# BOUND and add margin before an expensive run.
sel_fps = WeightedFPSRefs(
    p_gen, distance=D2, pool_size=FPS_POOL, start=0,
    points_per_cell=FPS_PPC, est_floor=FPS_FLOOR,
    seed=SEED, auto_r_grid=FPS_GRID, tau_target=TAU_TARGET,
    probe_size=PROBE_SIZE, fallback="best",
)

t0 = time.time()
R_fps = sel_fps.select(None).shape[0]
dt_fps = time.time() - t0

print(f"WeightedFPSRefs auto-R -> R = {R_fps}   ({dt_fps / 60:.1f} min, "
      f"projected {fps_passes * S_PASS / 60:.0f} min)   [treat as a LOWER BOUND]")
print(f"{'R':>4} {'tau(f_R, f_2R)':>15}  {'>= target':>9}")
for R, tau in sel_fps.tau_history:
    print(f"{R:4d} {tau:15.4f}  {'yes' if tau >= TAU_TARGET else 'no':>9}")
ceiling_fps = max((t for _, t in sel_fps.tau_history), default=float("nan"))
print(f"\nobserved tau ceiling: {ceiling_fps:.4f}")

w = sel_fps.weights
w_list = None
if w is not None:
    w = w.detach().cpu()
    w_list = [float(x) for x in w]
    ratio = float(w.max()) / max(float(w.min()), 1e-12)
    print(f"\nVoronoi weights (R={R_fps}): min={float(w.min()):.4f} max={float(w.max()):.4f} "
          f"max/min={ratio:.1f}")
    print("  A large max/min means p-mass is very unevenly spread over the FPS cells, i.e. plain")
    print("  (unweighted) FPS would have badly over-represented sparse regions of p.")
    print(f"  weights: {[round(x, 4) for x in w_list]}")

rho = list(getattr(sel_fps, "covering_radius_history", []) or [])
if rho:
    print("\ncovering radius rho(R) (max-min IEM distance; should fall as R grows):")
    print(f"  {[round(float(r), 3) for r in rho[:2 * max(FPS_GRID)]]}")

save("weighted_fps", {"R": R_fps, "tau_history": sel_fps.tau_history, "ceiling": ceiling_fps,
                      "minutes": dt_fps / 60, "weights": w_list,
                      "covering_radius": [float(r) for r in rho] if rho else None,
                      "grid": list(FPS_GRID), "pool": FPS_POOL})

In [ ]:
# --- comparison -----------------------------------------------------------------------------------
# The two selectors ran on DIFFERENT ladders (RandomRefs wider, since it is cheaper and unbiased), so
# compare them only where the ladders overlap -- R=4 and R=8.
ALL_R = sorted(set(RAND_GRID) | set(FPS_GRID))
print(f"{'selector':>16} {'auto-R':>7} {'ceiling':>8} {'min':>6}   tau by R")
for tag, r in results.items():
    hist = " ".join(f"{R}:{t:.3f}" for R, t in r["tau_history"])
    print(f"{tag:>16} {r['R']:7d} {r['ceiling']:8.4f} {r['minutes']:6.1f}   {hist}")

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for tag, r in results.items():
    Rs = [R for R, _ in r["tau_history"]]
    ts = [t for _, t in r["tau_history"]]
    ax.plot(Rs, ts, "o-", label=f"{tag} (auto-R = {r['R']})")
ax.axhline(TAU_TARGET, color="crimson", ls="--", lw=1, label=f"tau_target = {TAU_TARGET}")
ax.axvline(4, color="grey", ls=":", lw=1, label="R = 4 (the sweeps' choice)")
# tau resolution at this probe count -- differences smaller than this band are not meaningful.
se = (2 * (2 * PROBE_SIZE + 5) / (9 * PROBE_SIZE * (PROBE_SIZE - 1))) ** 0.5
ax.text(0.02, 0.03, f"tau SE $\\approx${se:.3f} at {PROBE_SIZE} probes\n(read the curve, not one value)",
        transform=ax.transAxes, fontsize=8, va="bottom", color="dimgrey")
ax.set_xscale("log", base=2)
ax.set_xticks(ALL_R); ax.set_xticklabels([str(r) for r in ALL_R])
ax.set_xlabel("R"); ax.set_ylabel(r"weighted-$\tau(f_R,\ f_{2R})$")
ax.set_title(f"Rank stability of $f$ vs reference count   ($d$={LATENT_DIM}, "
             f"N_GAMMA={N_GAMMA}, NUM_EPS={NUM_EPS})")
ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "auto_r_tau.png"), dpi=140, bbox_inches="tight")
plt.show()

# --- verdict --------------------------------------------------------------------------------------
print("\n" + "=" * 78)
t4 = {tag: dict(r["tau_history"]).get(4) for tag, r in results.items()}
stable = {tag: (v is not None and v >= TAU_TARGET) for tag, v in t4.items()}
for tag, v in t4.items():
    if v is not None:
        print(f"{tag}: tau at R=4 = {v:.4f} -> {'adequate' if stable[tag] else 'DISTORTING f'}")
if stable and all(stable.values()):
    print("\nVERDICT: R=4 already ranks f stably under both selectors. References are NOT the")
    print("  bottleneck -- spend the budget on GUIDANCE (which sets how wide p is, and is free) and")
    print("  on N_GAMMA (whose d-gamma weighting currently favours fine texture over structure),")
    print("  rather than on the expensive R increase.")
else:
    need = max((r["R"] for r in results.values()), default=None)
    print(f"\nVERDICT: R=4 does NOT rank f stably. Smallest adequate R across selectors: {need}")
    print("  (WeightedFPS's R is a LOWER BOUND -- add margin.) The R increase is now justified by")
    print("  measurement. Cost consequence at 24 sweeps/level: R=8 buys ~16.7 levels in 24 h,")
    print("  R=16 only ~8.8, and R=32 does not reach beta=1 in any allocation available here.")
if results and max(r["ceiling"] for r in results.values()) < TAU_TARGET:
    print(f"\nNOTE: no R reached tau_target={TAU_TARGET}; the ceiling itself is the finding. CLAUDE.md")
    print("  warns achievable tau falls in high d, so calibrate tau_target to this curve rather than")
    print(f"  assuming {TAU_TARGET} is reachable at d={LATENT_DIM}.")
print(f"\nReminder: tau SE ~{se:.3f} at {PROBE_SIZE} probes, so treat gaps under ~{se:.2f} as ties.")
print("=" * 78)
print(f"\ntotal notebook time: {(time.time() - T_START) / 60:.1f} min")

## Takeaways

*(Fill in after the run — this cell is the authoritative statement of what was learned, per repo convention.)*

**What this decides.** Whether the FLUX sweeps' `R = 4` was distorting the reward, and therefore whether the
next run should spend its budget on more references or on something else. The answer is a measurement, not a
judgement call: τ is the rank correlation between $f$ computed with R references and with 2R.

**To record:**

- τ at R = 4 for each selector, and whether it clears `tau_target`
- The τ **ceiling** — if no R reaches 0.97, that ceiling is itself the result, and `tau_target` should be
  recalibrated to it rather than assumed reachable at $d = 65536$
- The auto-chosen R from each selector, remembering `WeightedFPSRefs`' is an optimistic **lower bound**
- The Voronoi weight spread (`max/min`): large means $p$-mass is very unevenly distributed over FPS cells, so
  plain unweighted FPS would over-represent sparse regions
- Whether the two selectors agree. They fail differently — `RandomRefs` is unbiased with $1/R$ variance,
  `WeightedFPSRefs` is consistent but carries a within-cell quantization bias — so agreement is real evidence
  and disagreement localises the problem

**Then:**

- **R=4 adequate** → references are not the bottleneck. Spend on `GUIDANCE` (free, and it sets how wide $p$ is
  — the manifold the tilt is confined to) and `N_GAMMA`.
- **R=4 inadequate** → the expensive increase is justified. Budget consequence at 24 sweeps/level: R=8 buys
  ~16.7 levels in 24 h, R=16 only ~8.8, and R=32 does not reach β=1 in any allocation available here.

**Caveats.** Every auto-R control is scaled far below its default (probes 16 vs 128, draws 3 vs 20, FPS pool 32
vs 1000) purely for cost — the defaults would be 120 h and 558 h. So τ here is itself a noisy estimate of a
noisy quantity; treat differences under ~0.02 as ties. `N_GAMMA`/`NUM_EPS` are pinned to the sweep's 6/3 so the
answer applies to *that* reward, not a better-resolved one.